# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nadim7009/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Look before deciding. The heavy tails below are why this lane uses medians, a 500-impression floor and weighted errors instead of means.

In [1]:
# Setup: works in Colab (clones my repo) and locally (walks up to the repo root)
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/nadim7009/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold, KFold
from sklearn.ensemble import HistGradientBoostingRegressor
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
pool = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 500)].reset_index(drop=True)
print(f"{len(df):,} pages in the file; {len(pool):,} in my lane's pool from {pool['client_id'].nunique()} clients")

v = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 500)].copy()   # all positions with real volume
rows = []
for col in ["impressions_90d", "clicks_90d", "ctr", "avg_position", "content_age_days", "word_count"]:
    s = v[col].dropna()
    rows.append([col, len(s), s.median(), s.quantile(.99), s.max(), s.skew(), (s == 0).mean()])
dist = pd.DataFrame(rows, columns=["column", "n", "median", "p99", "max", "skew", "share_zero"]).round(2)
print(f"Pages with a position and >= 500 impressions: {len(v):,}")
print(dist.to_string(index=False))

30,000 pages in the file; 12,023 in my lane's pool from 28 clients
Pages with a position and >= 500 impressions: 16,726
          column     n  median       p99       max  skew  share_zero
 impressions_90d 16726 2948.50 100714.50 517715.00  8.96        0.00
      clicks_90d 16726    5.00    390.00   4178.00 14.07        0.15
             ctr 16726    0.17      1.49      5.43  3.37        0.15
    avg_position 16726   11.25     60.50     88.90  1.63        0.00
content_age_days 16726  236.00    545.00    557.00  0.46        0.00
      word_count 11977 2970.00   7635.96   9546.00  1.07        0.00


**What the distributions show (observed).** Impressions and clicks are heavily right-skewed (skew about 9 and 14): the median page has about 2,900 impressions and 5 clicks while the 99th percentile page has about 100,000 impressions. About 15% of these pages have zero clicks. `ctr` is skewed too, with a median of 0.17%. So a mean would describe almost nobody, and one huge page can move any pooled average. Everything below uses medians and shows how many pages sit in each bucket.

## 2. Signal test #1 / #2 / #3 (verdict each)

Rule for the verdicts, set before looking: a bucket needs **50 or more pages** to count. CONFIRMED means the claim holds in order across buckets. OPPOSITE means the order is reversed. MIXED means it holds in part. FALSE means the buckets do not separate.

**Signal 1: "pages ranked higher get a higher CTR."**

In [2]:
def table(col, order=None, min_n=50):
    t = v.groupby(col, observed=True)["ctr"].agg(n="size", median_ctr="median").round(3)
    if order: t = t.reindex(order)
    t["counts"] = np.where(t["n"] >= min_n, "ok", "too few, ignore")
    return t

order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
s1 = table("position_tier", order)
print(s1.to_string())
ok = s1[s1["n"] >= 50]["median_ctr"].values
mono = bool(np.all(np.diff(ok) <= 0))
print(f"\nMedian CTR falls in order across the {len(ok)} usable tiers: {mono}")

                  n  median_ctr counts
position_tier                         
top_3           458        0.20     ok
page_1         7064        0.24     ok
striking       4485        0.17     ok
page_3_5       4330        0.09     ok
deep            389        0.00     ok

Median CTR falls in order across the 5 usable tiers: False


**Verdict, signal 1: MIXED.** CTR does fall as pages move down the results (page 1 about 0.24%, `deep` about 0.00%), but the very top tier (`top_3`) sits *below* page 1, which is backwards, and inside one tier the spread is wide. Position matters and does not decide CTR on its own.

**Signal 2: "longer pages get a higher CTR."**

In [3]:
order2 = ["<1000", "1000-2000", "2000-3500", "3500+"]
s2 = table("word_count_tier", order2)
print(s2.to_string())
ok2 = s2[s2["n"] >= 50]["median_ctr"].values
print(f"\nMedian CTR rises with length across the usable tiers: {bool(np.all(np.diff(ok2) >= 0))}")
print(f"Pages with no word count at all: {v['word_count'].isna().mean():.0%}")

                    n  median_ctr           counts
word_count_tier                                   
<1000               9        1.37  too few, ignore
1000-2000        1306        0.14               ok
2000-3500        6824        0.21               ok
3500+            3838        0.16               ok

Median CTR rises with length across the usable tiers: False
Pages with no word count at all: 28%


**Verdict, signal 2: MIXED, leaning FALSE.** The 2,000 to 3,500 word band has the highest median CTR (0.21%) and the 3,500+ band drops back (0.16%), so more words is not simply better. The under-1,000 tier has 9 pages and is ignored. About 28% of these pages have no word count, so this signal describes the pages that have one.

**Signal 3: "search intent separates CTR."**

In [4]:
s3 = table("main_intent")
print(s3.to_string())
usable = s3[s3["n"] >= 50]["median_ctr"]
print(f"\nSpread of medians across usable intents: {usable.min():.2f}% to {usable.max():.2f}%  "
      f"(the position tiers above span {s1[s1['n'] >= 50]['median_ctr'].max() - s1[s1['n'] >= 50]['median_ctr'].min():.2f} points)")

                   n  median_ctr           counts
main_intent                                      
commercial      2841        0.16               ok
informational  10094        0.16               ok
navigational      19        0.27  too few, ignore
transactional   3561        0.19               ok

Spread of medians across usable intents: 0.16% to 0.19%  (the position tiers above span 0.24 points)


**Verdict, signal 3: FALSE as a standalone signal.** Informational and commercial pages both have a median CTR of 0.16%, and transactional is 0.19%. The gap between intents is a few hundredths of a point, much smaller than the gap between position tiers. Intent may still help inside a model that already knows position, which is a different question. `navigational` has 19 pages and is ignored.

## 3. The flag-linked test

FlyRank's refresh flag assumes that **pages that have gone longest without an update are the ones declining**. I test that assumption directly: the share of pages trending down, by days since the last update.

In [5]:
d = df.copy()
d["declining"] = (d["trend_direction"] == "down").astype(int)
d["stale_tier"] = pd.cut(d["days_since_last_update"], [-1, 30, 90, 180, 100000], labels=["0-30", "31-90", "91-180", "181+"])
st = d.groupby("stale_tier", observed=True).agg(pages=("declining", "size"), share_declining=("declining", "mean")).round(3)
st["counts"] = np.where(st["pages"] >= 50, "ok", "too few, ignore")
print(st.to_string())
print(f"\nShare trending down overall: {d['declining'].mean():.3f}")

            pages  share_declining counts
stale_tier                               
0-30        20480            0.511     ok
31-90         175            0.589     ok
91-180       9171            0.611     ok
181+          174            0.471     ok

Share trending down overall: 0.542


**Verdict: MIXED, weak.** Across the whole file the share of pages trending down is 0.51 for pages updated in the last 30 days and 0.61 for pages at 91 to 180 days, so older pages do decline somewhat more. It is not a clean climb: the `181+` tier is the lowest at 0.47, and the 31-90 and 181+ tiers hold only about 175 pages each. Restricted to the visible pool my lane uses, the two big tiers are closer still (0.59 and 0.62, in the week-7 notebook). So the flag's assumption has weak support at best, and a gap of about 10 points is not enough to build a rule on. This is a measured result about this slice and not a claim about the flag as a whole. (It is the same check behind keeping staleness out of my week-4 rule.)

## 4. What this means in practice

A content team should treat position as a starting point, not a verdict: a page's CTR has to be read against its own tier, and `top_3` rows need a second look. Page length and search intent do not predict CTR well enough to be a rule, and "refresh the oldest pages first" has only weak support in this file. So a review queue built on the CTR gap against the page's tier, with a person checking each flagged page, is a better use of time than a queue built on age or length. All of this is observed on one 90-day slice and is directional.

## Self-check

- [x] Every section above is filled, with the thinking in markdown and the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime, then Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit the repo URL on the card.